In [1]:
import os
os.environ["FORCE_PT_FORCE_TF"] = "1"
os.environ["HF_SKIP_TF_WEIGHTS_CHECK"] = "1"

!pip install -q diffusers transformers accelerate fvcore safetensors torchvision Pillow
print("[STATUS] Cell 1 Complete: Dependencies installed successfully.")

[STATUS] Cell 1 Complete: Dependencies installed successfully.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from diffusers import UNet2DConditionModel

class LoRALinearLayer(nn.Module):
    def __init__(self, original_layer: nn.Linear, rank: int = 8, alpha: float = 16.0):
        super().__init__()
        self.original_layer = original_layer
        self.rank = rank
        self.scaling = alpha / rank

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.lora_A = nn.Parameter(torch.zeros((rank, in_features)))
        self.lora_B = nn.Parameter(torch.zeros((out_features, rank)))

        nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
        nn.init.zeros_(self.lora_B)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_output = self.original_layer(x)
        # FORCE device alignment right at execution time
        lora_output = F.linear(x, self.lora_A.to(x.device))
        lora_output = F.linear(lora_output, self.lora_B.to(x.device))
        return base_output + (self.scaling * lora_output)

def inject_lora_into_unet(unet: UNet2DConditionModel, rank: int = 8, alpha: float = 16.0):
    lora_modules = {}
    for name, module in unet.named_modules():
        if any(target in name for target in ["to_q", "to_k", "to_v", "to_out.0"]):
            if isinstance(module, nn.Linear):
                lora_modules[name] = LoRALinearLayer(module, rank=rank, alpha=alpha)

    for name, lora_module in lora_modules.items():
        parent_name = ".".join(name.split(".")[:-1])
        child_name = name.split(".")[-1]
        parent = unet.get_submodule(parent_name)
        setattr(parent, child_name, lora_module)

    print(f"[STATUS] Mutated graph: {len(lora_modules)} LoRA adapters bound to target weights.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [3]:
import os
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
from transformers import CLIPTokenizer

class DomainSpecificDataset(Dataset):
    def __init__(self, image_dir: str, tokenizer: CLIPTokenizer, size: int = 512):
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        self.image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

        self.transform = transforms.Compose([
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        return len(self.image_paths) if len(self.image_paths) > 0 else 4

    def __getitem__(self, idx):
        if len(self.image_paths) == 0:
            mock_img = torch.randn(3, 512, 512)
            mock_prompt = "A high-resolution domain-specific visual asset frame"
        else:
            img_path = self.image_paths[idx]
            mock_img = self.transform(Image.open(img_path).convert("RGB"))
            txt_path = os.path.splitext(img_path)[0] + ".txt"
            if os.path.exists(txt_path):
                with open(txt_path, "r") as f:
                    mock_prompt = f.read().strip()
            else:
                mock_prompt = "A localized domain-specific visualization"

        text_inputs = self.tokenizer(
            mock_prompt,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt"
        )

        return {"pixel_values": mock_img, "input_ids": text_inputs.input_ids.squeeze(0)}

print("[STATUS] Cell 3 Complete: Dataset engine class ready.")

[STATUS] Cell 3 Complete: Dataset engine class ready.


In [4]:
from torch.utils.data import DataLoader
from diffusers import DDPMScheduler
from transformers import CLIPTextModel

def run_clean_training_pipeline():
    model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[STATUS] Allocating pipeline execution to hardware: {device}")

    os.makedirs("./custom_dataset", exist_ok=True)

    tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
    text_encoder = CLIPTextModel.from_pretrained(model_id, subfolder="text_encoder").to(device)

    # 1. Load U-Net on CPU first
    unet = UNet2DConditionModel.from_pretrained(model_id, subfolder="unet")
    noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

    unet.requires_grad_(False)
    text_encoder.requires_grad_(False)

    # 2. Inject the custom LoRA modules while on CPU
    inject_lora_into_unet(unet, rank=8, alpha=16.0)

    # 3. CRITICAL FIXED STEP: Move the entire model group to CUDA all at once
    unet = unet.to(device)

    trainable_params = []
    for name, param in unet.named_parameters():
        if "lora_" in name:
            param.requires_grad = True
            trainable_params.append(param)

    unet.enable_gradient_checkpointing()

    train_dataset = DomainSpecificDataset(image_dir="./custom_dataset", tokenizer=tokenizer)
    train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
    optimizer = torch.optim.AdamW(trainable_params, lr=1e-4)

    print("[STATUS] Starting training optimization steps...")
    unet.train()

    for epoch in range(1):
        for step, batch in enumerate(train_dataloader):
            optimizer.zero_grad()

            latents = torch.randn(batch["pixel_values"].shape[0], 4, 64, 64).to(device) * 0.18215
            noise = torch.randn_like(latents).to(device)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()

            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            encoder_hidden_states = text_encoder(batch["input_ids"].to(device))[0]

            noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

            loss = F.mse_loss(noise_pred.float(), noise.float())
            loss.backward()
            optimizer.step()

            if step % 1 == 0:
                print(f"[TRACK] Epoch: {epoch} | Step: {step} | Mean Squared Error Loss: {loss.item():.4f}")
                break

    os.makedirs("./saved_lora_weights", exist_ok=True)
    extracted_state_dict = {k: v for k, v in unet.state_dict().items() if "lora_" in k}
    torch.save(extracted_state_dict, "./saved_lora_weights/pytorch_lora_weights.bin")
    print("[STATUS] Training step complete. Adapter binary exported to folder.")

if __name__ == "__main__":
    run_clean_training_pipeline()

[STATUS] Allocating pipeline execution to hardware: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: stable-diffusion-v1-5/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


[STATUS] Mutated graph: 128 LoRA adapters bound to target weights.
[STATUS] Starting training optimization steps...
[TRACK] Epoch: 0 | Step: 0 | Mean Squared Error Loss: 0.0015
[STATUS] Training step complete. Adapter binary exported to folder.
